# 01. Experimental Data Generation

This notebook builds the controlled dataset used in the ECDSA ? side-channel leakage ? CNN ? HNP experimental pipeline.

The experiment uses the secp256k1 curve, 160 synthetic signatures, 12 MSB bits treated as leaked for each nonce, and Gaussian noise with ? = 0.15 for the profiled dataset. The private key and complete nonces remain in the oracle and do not enter the public data used by the prediction stages.

## Scope and Experimental Question

The project tests whether discrete information about the ECDSA nonce can be estimated from noisy synthetic traces and then transformed into an HNP instance.

For an ECDSA signature, with private key $d$, nonce $k$, message hash $z$, and group order $n$:

$$
s \equiv k^{-1}(z + rd) \pmod n.
$$

If the first $\ell$ bits of $k$ are known, we can write:

$$
k = a + \delta, \qquad
a = \bar{k}\,2^{256-\ell}, \qquad
0 \le \delta < 2^{256-\ell}.
$$

The notebook works only with locally generated data. No keys, messages, or nonces obtained from external systems are used.

## Experiment Configuration

| Parameter | Value | Role |
| --- | ---: | --- |
| Curve | secp256k1 | simulated ECDSA group |
| Number of samples | 160 | independent signatures |
| Leaked bits | 12 | MSB prefix used later in HNP |
| Leakage model | HW | Hamming Weight on the processed prefix |
| Noise | ? = 0.15 | profiled dataset for CNN |
| Split | 96 train, 24 validation, 40 test | identical to the CNN and HNP pipeline |
| Seed | 20260919 | reproducibility |

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "informatica").is_dir() and (p / "notebooks").is_dir()
)

ECDSA_DIR = PROJECT_ROOT / "informatica" / "ECDSA"
CNN_DIR = PROJECT_ROOT / "informatica" / "cnn"
DATASET_DIR = PROJECT_ROOT / "informatica" / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "informatica" / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"

for path in (ECDSA_DIR, CNN_DIR, DATASET_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from generate import generate
from dataset_generator3 import CURVE, load_public_dataset, load_ground_truth, is_on_curve, scalar_mult, INF
from ecdsa_leakage_model4 import generate_clean_leakage, integer_to_bits

## Dataset Generation

The function writes the public data, oracle, profiling dataset, metadata, and train/validation/test split. The configuration uses 96 samples for training, 24 for validation, and 40 for testing. The local run writes the complete private key to `oracle_dataset.json`; before publication, the `private_key` field value must be redacted as `<REDACTED_PRIVATE_KEY>`. Do not publish the real private key.


In [ ]:
metadata = generate(
    sample_count=160,
    leaked_bits=12,
    noise_sigma=0.15,
    seed=20260919,
    leakage_mode="HW",
    train_fraction=0.60,
    validation_fraction=0.15,
)

split = json.loads((ARTIFACTS_DIR / "dataset_split.json").read_text(encoding="utf-8"))
split_sizes = {name: len(split[name]) for name in ("train", "validation", "test")}
assert split_sizes == {"train": 96, "validation": 24, "test": 40}
display(pd.DataFrame([metadata]).T.rename(columns={0: "valoare"}))
display(pd.DataFrame([split_sizes], index=["eșantioane"]).T.rename(columns={0: "număr"}))

## Curve and Signature Verification

Before the statistical analysis, we verify generator $G$, the subgroup order, and consistency between the public signature and the nonce retained in the oracle. The tests check only internal properties of the simulation.

In [ ]:
public_records = load_public_dataset()
oracle = load_ground_truth()

curve_ok = is_on_curve(CURVE, CURVE.G)
order_ok = scalar_mult(CURVE, CURVE.n, CURVE.G) == INF

truth_by_id = {int(k): v for k, v in oracle["samples"].items()}
leakage_ok = True
sample_ok = len(public_records) == len(truth_by_id) == 160

for record in public_records:
    sample_id = int(record["sample_id"])
    truth = truth_by_id[sample_id]
    nonce = int(truth["nonce_k"])
    leaked = int(record["leaked_bits"])
    hidden_bits = CURVE.n.bit_length() - leaked
    expected_prefix = int(truth["leaked_nonce_prefix"])
    leakage_ok &= (nonce >> hidden_bits) == expected_prefix

checks = pd.DataFrame([
    ["Generatorul este pe curbă", curve_ok],
    ["nG este punctul la infinit", order_ok],
    ["160 eșantioane publice și oracle", sample_ok],
    ["Prefixele MSB sunt coerente cu nonce-urile", leakage_ok],
], columns=["verificare", "rezultat"])

display(checks)
assert checks["rezultat"].all()

## Public Data Structure

The public data contain the information available to the prediction stages: the message, hash $z$, signature components $r$ and $s$, and the number of bits considered available.

In [ ]:
public_df = pd.DataFrame(public_records)
public_df["z_hex"] = public_df["z"].map(lambda x: f"{int(x):064x}")
public_df["r_hex"] = public_df["r"].map(lambda x: f"{int(x):064x}")
public_df["s_hex"] = public_df["s"].map(lambda x: f"{int(x):064x}")

display(public_df[["sample_id", "message", "leaked_bits", "z_hex", "r_hex", "s_hex"]].head(6))
display(public_df[["z", "r", "s", "leaked_bits"]].describe().T)

## Oracle Separation

The private key and complete nonces serve as control data. The baseline and CNN must not read these values to produce predictions; the oracle participates only in generation and result evaluation.

In [ ]:
profile_records = json.loads((ARTIFACTS_DIR / "profile_dataset.json").read_text(encoding="utf-8"))

separation = pd.DataFrame([
    ["public_dataset.json", len(public_records), "z, r, s, leaked_bits", "fără cheia privată și fără nonce complet"],
    ["oracle_dataset.json", len(truth_by_id), "cheie privată, nonce, prefix real", "control experimental"],
    ["profile_dataset.json", len(profile_records), "trace, bit_labels, z, r, s", "input pentru model"],
], columns=["fișier", "eșantioane", "câmpuri", "utilizare"])

display(separation)

## A Synthetic Trace

The HW model calculates the Hamming weight of the processed nonce prefix at each position. At each position, we add independent Gaussian noise.

$$
HW(v)=\operatorname{popcount}(v), \qquad
y_j = HW(k_{\le j}) + arepsilon_j, \qquad
arepsilon_j \sim \mathcal N(0,\sigma^2).
$$

In [ ]:
first_id = 0
first_truth = truth_by_id[first_id]
first_nonce = int(first_truth["nonce_k"])

clean = generate_clean_leakage(first_nonce, CURVE.n.bit_length(), mode="HW", alpha=1.0)
noisy = profile_records[first_id]["trace"]

fig = plt.figure(figsize=(11, 4.5))
x = np.arange(1, len(clean) + 1)
plt.plot(x, clean, label="urmă curată")
plt.plot(x, noisy, label="urmă zgomotoasă", alpha=0.75)
plt.xlim(1, 32)
plt.xlabel("Poziția bitului procesat")
plt.ylabel("Valoarea scurgerii")
plt.title("Primul eșantion: model HW")
plt.legend()
plt.tight_layout()
plt.show()

## Verification of CNN Labels

The labels in the profiling dataset must match the nonce bits in MSB ? LSB order. In the current experiment, the CNN consumes the first 12 positions, i.e. the prefix used by the HNP stage.

In [ ]:
label_checks = []
for record in profile_records:
    sample_id = int(record["sample_id"])
    nonce = int(truth_by_id[sample_id]["nonce_k"])
    expected = integer_to_bits(nonce, CURVE.n.bit_length())[:12]
    observed = list(map(int, record["bit_labels"][:12]))
    label_checks.append(expected == observed)

label_accuracy = float(np.mean(label_checks))
print(f"Concordanța etichetelor de 12 biți: {100 * label_accuracy:.2f}%")
assert label_accuracy == 1.0

## Produced Artifacts

Running the notebook creates the dataset used by the other notebooks. The oracle file remains separate from the public data and the model inputs.

In [ ]:
generated_files = [
    "public_dataset.json",
    "oracle_dataset.json",
    "profile_dataset.json",
    "dataset_metadata.json",
    "dataset_split.json",
]

display(pd.DataFrame({
    "fișier": generated_files,
    "existent": [ (ARTIFACTS_DIR / name).exists() for name in generated_files ],
    "dimensiune_bytes": [ (ARTIFACTS_DIR / name).stat().st_size if (ARTIFACTS_DIR / name).exists() else 0 for name in generated_files ],
}))

## References

- T. Pornin, [RFC 6979](https://www.rfc-editor.org/rfc/rfc6979), for nonce generation and ECDSA context.
- Bitcoin Core, [secp256k1 parameters](https://github.com/bitcoin-core/secp256k1/blob/master/sage/secp256k1_params.sage).
- Implementation used in the project: `informatica/ECDSA/dataset_generator3.py` and `informatica/dataset/generate.py`.